# Part 1: Data Audit, EDA & Business Understanding
This notebook loads all raw datasets, performs quality checks,
exploratory analysis, and identifies churn-risk patterns.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# ---- Load all files ----
customers = pd.read_csv('../data/raw/customers.csv')
orders    = pd.read_csv('../data/raw/orders.csv')
tickets   = pd.read_csv('../data/raw/support_tickets.csv')
events    = pd.read_csv('../data/raw/web_app_events.csv')
churn     = pd.read_csv('../data/raw/churn_labels.csv')
campaigns = pd.read_csv('../data/raw/campaigns_interventions.csv')
snapshot  = pd.read_csv('../data/raw/modeling_snapshot.csv')

# ---- Quick size check ----
datasets = {
    'customers': customers, 'orders': orders, 'tickets': tickets,
    'events': events, 'churn': churn, 'campaigns': campaigns,
    'snapshot': snapshot
}

for name, df in datasets.items():
    print(f"{name:15s} → rows: {df.shape[0]:,}  cols: {df.shape[1]}  columns: {list(df.columns)}")


In [ ]:
# Run this for each dataset one by one
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"DATASET: {name.upper()}")
    print(f"{'='*60}")
    print(df.dtypes)
    print("\nFirst 3 rows:")
    display(df.head(3))
    print("\nBasic stats:")
    display(df.describe(include='all'))


In [ ]:
# ---- Missing values ----
print("=== MISSING VALUES ===")
for name, df in datasets.items():
    missing = df.isnull().sum()
    pct = (df.isnull().mean() * 100).round(2)
    summary = pd.DataFrame({'missing_count': missing, 'missing_pct': pct})
    summary = summary[summary['missing_count'] > 0]
    if len(summary) > 0:
        print(f"\n{name}:")
        print(summary.to_string())

# ---- Duplicate rows ----
print("\n=== DUPLICATE ROWS ===")
for name, df in datasets.items():
    dupes = df.duplicated().sum()
    print(f"{name}: {dupes} duplicate rows")

# ---- Duplicate IDs (should be unique keys) ----
print("\n=== DUPLICATE CUSTOMER IDs ===")
print("customers:", customers['customer_id'].duplicated().sum())
print("orders:   ", orders['order_id'].duplicated().sum() if 'order_id' in orders.columns else "no order_id")
print("churn:    ", churn['customer_id'].duplicated().sum())

# ---- Negative values ----
print("\n=== NEGATIVE ORDER VALUES (possible refunds) ===")
if 'order_value' in orders.columns:
    neg = orders[orders['order_value'] < 0]
    print(f"Rows with negative order_value: {len(neg)}")
    print(neg.head())

# ---- Date sanity check ----
print("\n=== DATE RANGES ===")
orders['order_date'] = pd.to_datetime(orders['order_date'])
print("Orders date range:", orders['order_date'].min(), "to", orders['order_date'].max())

# ---- Join integrity ----
print("\n=== JOIN KEY INTEGRITY ===")
orders_no_customer = set(orders['customer_id']) - set(customers['customer_id'])
churn_no_customer  = set(churn['customer_id'])  - set(customers['customer_id'])
print(f"Orders with no customer record: {len(orders_no_customer)}")
print(f"Churn labels with no customer record: {len(churn_no_customer)}")

# ---- Outliers ----
print("\n=== OUTLIERS IN ORDER VALUE ===")
Q1 = orders['order_value'].quantile(0.25)
Q3 = orders['order_value'].quantile(0.75)
IQR = Q3 - Q1
outliers = orders[
    (orders['order_value'] < Q1 - 1.5*IQR) |
    (orders['order_value'] > Q3 + 1.5*IQR)
]
print(f"Outlier rows: {len(outliers)}")
print(f"Max order value: {orders['order_value'].max():.2f}")
print(f"Normal range (IQR): {Q1:.2f} to {Q3:.2f}")


In [ ]:
# ==============================
# CHART 1: Churn distribution
# ==============================
plt.figure(figsize=(6, 4))
churn['churned'].value_counts().plot(kind='bar', color=['#4C72B0', '#DD8452'], edgecolor='white')
plt.title('Customer Churn Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Churned (0=No, 1=Yes)')
plt.ylabel('Count')
plt.xticks(rotation=0)
churn_pct = churn['churned'].mean() * 100
plt.text(0.5, 0.95, f"Churn rate: {churn_pct:.1f}%", transform=plt.gca().transAxes,
         ha='center', fontsize=11, color='gray')
plt.tight_layout()
plt.savefig('charts/01_churn_distribution.png', dpi=150)
plt.show()

# ==============================
# CHART 2: Order value distribution
# ==============================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
orders['order_value'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Order Value Distribution')
axes[0].set_xlabel('Order Value')
np.log1p(orders['order_value'].clip(lower=0)).hist(bins=50, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Order Value (log scale)')
axes[1].set_xlabel('log(1 + Order Value)')
plt.tight_layout()
plt.savefig('charts/02_order_value_distribution.png', dpi=150)
plt.show()

# ==============================
# CHART 3: Monthly orders trend
# ==============================
orders['order_month'] = orders['order_date'].dt.to_period('M')
monthly = orders.groupby('order_month').size()
plt.figure(figsize=(12, 4))
monthly.plot(color='steelblue', linewidth=2)
plt.title('Monthly Order Volume Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Number of Orders')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('charts/03_monthly_orders.png', dpi=150)
plt.show()

# ==============================
# CHART 4: Support ticket types
# ==============================
if 'issue_type' in tickets.columns:
    plt.figure(figsize=(10, 4))
    tickets['issue_type'].value_counts().plot(kind='barh', color='steelblue')
    plt.title('Support Ticket Issue Types', fontsize=14, fontweight='bold')
    plt.xlabel('Count')
    plt.tight_layout()
    plt.savefig('charts/04_ticket_types.png', dpi=150)
    plt.show()

# ==============================
# CHART 5: Recency vs Churn (KEY CHART)
# ==============================
snapshot_date = orders['order_date'].max()  # use last date in data as snapshot
recency = orders.groupby('customer_id')['order_date'].max().reset_index()
recency.columns = ['customer_id', 'last_order']
recency['recency_days'] = (snapshot_date - recency['last_order']).dt.days

merged = recency.merge(churn, on='customer_id')

plt.figure(figsize=(8, 5))
sns.boxplot(x='churned', y='recency_days', data=merged,
            palette={0: '#4C72B0', 1: '#DD8452'})
plt.title('Recency (Days Since Last Order) vs Churn', fontsize=14, fontweight='bold')
plt.xlabel('Churned (0=No, 1=Yes)')
plt.ylabel('Days Since Last Order')
median_churn    = merged[merged['churned']==1]['recency_days'].median()
median_retained = merged[merged['churned']==0]['recency_days'].median()
plt.text(0.98, 0.95,
         f"Median retained: {median_retained:.0f} days\nMedian churned: {median_churn:.0f} days",
         transform=plt.gca().transAxes, ha='right', fontsize=10, color='gray')
plt.tight_layout()
plt.savefig('charts/05_recency_vs_churn.png', dpi=150)
plt.show()

# ==============================
# CHART 6: Order frequency vs Churn
# ==============================
freq = orders.groupby('customer_id').size().reset_index(name='order_count')
merged_freq = freq.merge(churn, on='customer_id')

plt.figure(figsize=(8, 5))
sns.boxplot(x='churned', y='order_count', data=merged_freq,
            palette={0: '#4C72B0', 1: '#DD8452'})
plt.title('Order Frequency vs Churn', fontsize=14, fontweight='bold')
plt.xlabel('Churned (0=No, 1=Yes)')
plt.ylabel('Total Orders')
plt.tight_layout()
plt.savefig('charts/06_frequency_vs_churn.png', dpi=150)
plt.show()

# ==============================
# CHART 7: Support ticket count vs Churn rate
# ==============================
ticket_count = tickets.groupby('customer_id').size().reset_index(name='ticket_count')
merged_tickets = ticket_count.merge(churn, on='customer_id')
merged_tickets['ticket_bucket'] = pd.cut(
    merged_tickets['ticket_count'], bins=[0,1,2,3,5,100],
    labels=['0','1','2','3–5','5+'], right=False
)

churn_by_ticket = merged_tickets.groupby('ticket_bucket')['churned'].mean() * 100

plt.figure(figsize=(8, 4))
churn_by_ticket.plot(kind='bar', color='coral', edgecolor='white')
plt.title('Churn Rate by Support Ticket Count', fontsize=14, fontweight='bold')
plt.xlabel('Number of Support Tickets')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('charts/07_tickets_vs_churn_rate.png', dpi=150)
plt.show()

# ==============================
# CHART 8: Spend (monetary) vs Churn
# ==============================
spend = orders.groupby('customer_id')['order_value'].sum().reset_index(name='total_spend')
merged_spend = spend.merge(churn, on='customer_id')

plt.figure(figsize=(8, 5))
sns.boxplot(x='churned', y='total_spend', data=merged_spend,
            palette={0: '#4C72B0', 1: '#DD8452'})
plt.yscale('log')
plt.title('Total Spend vs Churn (log scale)', fontsize=14, fontweight='bold')
plt.xlabel('Churned (0=No, 1=Yes)')
plt.ylabel('Total Spend (log scale)')
plt.tight_layout()
plt.savefig('charts/08_spend_vs_churn.png', dpi=150)
plt.show()


In [ ]:
# ============================================================
# HYPOTHESIS 1: Low-recency customers churn more
# ============================================================
print("=== Hypothesis 1: Recency ===")
print(merged.groupby('churned')['recency_days'].describe())
# Expected: churned=1 group has much higher median recency_days

# ============================================================
# HYPOTHESIS 2: Customers with 2+ tickets churn more
# ============================================================
print("\n=== Hypothesis 2: Support Tickets ===")
print("Churn rate with 0 tickets:", 
      merged_tickets[merged_tickets['ticket_count']==0]['churned'].mean().round(3))
print("Churn rate with 2+ tickets:", 
      merged_tickets[merged_tickets['ticket_count']>=2]['churned'].mean().round(3))

# ============================================================
# HYPOTHESIS 3: Single-purchase customers churn at higher rate
# ============================================================
print("\n=== Hypothesis 3: Frequency ===")
single = merged_freq[merged_freq['order_count']==1]['churned'].mean()
multi  = merged_freq[merged_freq['order_count']>1]['churned'].mean()
print(f"Churn rate (1 order): {single:.3f}  vs  Churn rate (2+ orders): {multi:.3f}")

# ============================================================
# HYPOTHESIS 4: Non-campaign responders churn more
# ============================================================
print("\n=== Hypothesis 4: Campaign Engagement ===")
campaign_resp = campaigns.groupby('customer_id')['response'].max().reset_index()
campaign_resp.columns = ['customer_id', 'ever_responded']
merged_camp = campaign_resp.merge(churn, on='customer_id')
print(merged_camp.groupby('ever_responded')['churned'].mean())

# ============================================================
# HYPOTHESIS 5: High refund rate correlates with churn
# ============================================================
print("\n=== Hypothesis 5: Refund Rate ===")
refunds = orders[orders['order_value'] < 0].groupby('customer_id').size().reset_index(name='refund_count')
total_o = orders.groupby('customer_id').size().reset_index(name='total_orders')
refund_df = refunds.merge(total_o, on='customer_id')
refund_df['refund_rate'] = refund_df['refund_count'] / refund_df['total_orders']
merged_refund = refund_df.merge(churn, on='customer_id')
print(merged_refund.groupby('churned')['refund_rate'].describe())
